In [3]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 5, Finished, Available, Finished, False)

In [4]:
from pyspark.sql import SparkSession

spark

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 6, Finished, Available, Finished, False)

In [5]:
df_hospitals = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://Healthcare_Analytics_Project@onelake.dfs.fabric.microsoft.com/lh_healthcare_bronze.Lakehouse/Files/hospitals.csv")

display(df_hospitals)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d0a4d6a2-a40e-4543-a4fc-7da195da78d3)

In [6]:
df_hospitals.printSchema()

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 8, Finished, Available, Finished, False)

root
 |-- hospital_id: integer (nullable = true)
 |-- hospital_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- hospital_type: string (nullable = true)
 |-- bed_capacity: integer (nullable = true)
 |-- icu_beds: integer (nullable = true)
 |-- emergency_available: string (nullable = true)



In [7]:
display(df_hospitals)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 92ca89f5-d68f-499a-8445-82565c97a818)

<mark>**Clean Hospital Dataset**</mark>

In [8]:
from pyspark.sql.functions import (
    trim,
    when,
    col,
    upper,
    coalesce,
    lit
)

df_hospitals_clean = df_hospitals \
    .withColumn("hospital_name", trim(col("hospital_name"))) \
    .withColumn("city", trim(col("city"))) \
    .withColumn("state", trim(col("state"))) \
    .withColumn("hospital_type", trim(col("hospital_type"))) \
    .withColumn(
        "emergency_available",
        when(
            upper(trim(col("emergency_available"))).isin("TRUE", "YES"),
            "Yes"
        ).otherwise("No")
    ) \
    .withColumn(
        "icu_beds",
        coalesce(col("icu_beds"), lit(0))
    ) \
    .withColumn(
        "bed_capacity",
        when(col("bed_capacity") < 0, 0)
        .otherwise(col("bed_capacity"))
    )

# Maintain original CSV column order

df_hospitals_clean = df_hospitals_clean.select(
    "hospital_id",
    "hospital_name",
    "city",
    "state",
    "hospital_type",
    "bed_capacity",
    "icu_beds",
    "emergency_available"
)

display(df_hospitals_clean)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cad69457-b5b6-4f96-8b3d-26f1dfda67a8)

<mark>**departments.csv Read**</mark>

In [9]:
df_departments = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://Healthcare_Analytics_Project@onelake.dfs.fabric.microsoft.com/lh_healthcare_bronze.Lakehouse/Files/departments.csv")

display(df_departments)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3cd6afa9-f28f-4285-bd31-d764ff4a06bf)

In [10]:
df_departments.printSchema()

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 12, Finished, Available, Finished, False)

root
 |-- department_id: integer (nullable = true)
 |-- department_name: string (nullable = true)
 |-- department_category: string (nullable = true)



<mark>**Clean Departments Dataset**</mark>

In [11]:
from pyspark.sql.functions import (
    trim,
    col,
    coalesce,
    lit
)

df_departments_clean = df_departments \
    .withColumn(
        "department_name",
        trim(col("department_name"))
    ) \
    .withColumn(
        "department_category",
        trim(col("department_category"))
    ) \
    .withColumn(
        "department_category",
        coalesce(col("department_category"), lit("Unknown"))
    )

# Maintain original CSV column order

df_departments_clean = df_departments_clean.select(
    "department_id",
    "department_name",
    "department_category"
)

display(df_departments_clean)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 51cb5176-2ebb-4f9f-8385-42313d9639d4)

<mark>**doctors.csv Read**</mark>

In [12]:
df_doctors = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://Healthcare_Analytics_Project@onelake.dfs.fabric.microsoft.com/lh_healthcare_bronze.Lakehouse/Files/doctors.csv")

display(df_doctors)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b5b3077e-6a35-499b-8140-b810f096685a)

In [13]:
df_doctors.printSchema()

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 15, Finished, Available, Finished, False)

root
 |-- doctor_id: integer (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- department_id: integer (nullable = true)
 |-- hospital_id: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- experience_years: integer (nullable = true)
 |-- consultation_fee: integer (nullable = true)
 |-- shift_type: string (nullable = true)
 |-- employment_type: string (nullable = true)



<mark>**Cleaned Doctor Dataset**</mark>

In [14]:
from pyspark.sql.functions import (
    trim,
    when,
    upper,
    col,
    coalesce,
    lit
)

df_doctors_clean = df_doctors \
    .withColumn(
        "doctor_name",
        trim(col("doctor_name"))
    ) \
    .withColumn(
        "gender",
        when(
            upper(trim(col("gender"))) == "M",
            "Male"
        )
        .when(
            upper(trim(col("gender"))) == "F",
            "Female"
        )
        .otherwise(trim(col("gender")))
    ) \
    .withColumn(
        "shift_type",
        coalesce(trim(col("shift_type")), lit("Unknown"))
    ) \
    .withColumn(
        "employment_type",
        trim(col("employment_type"))
    ) \
    .withColumn(
        "consultation_fee",
        when(col("consultation_fee") < 0, 0)
        .otherwise(col("consultation_fee"))
    ) \
    .withColumn(
        "experience_years",
        when(col("experience_years") < 0, 0)
        .otherwise(col("experience_years"))
    )

# Maintain original CSV column order

df_doctors_clean = df_doctors_clean.select(
    "doctor_id",
    "doctor_name",
    "department_id",
    "hospital_id",
    "gender",
    "experience_years",
    "consultation_fee",
    "shift_type",
    "employment_type"
)

display(df_doctors_clean)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f985af9e-a43d-4318-8d03-96cf62b2c9b7)

<mark>**patients.csv Read**</mark>

In [15]:
df_patients = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://Healthcare_Analytics_Project@onelake.dfs.fabric.microsoft.com/lh_healthcare_bronze.Lakehouse/Files/patients.csv")

display(df_patients)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 551dfc38-80bb-40d4-b439-03e68f1c69db)

In [16]:
df_patients.printSchema()

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 18, Finished, Available, Finished, False)

root
 |-- patient_id: integer (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- date_of_birth: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- blood_group: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- phone_number: double (nullable = true)
 |-- email: string (nullable = true)



In [17]:
display(
    df_patients.select("date_of_birth", "registration_date")
)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 847c3b89-8230-4ba5-b6d0-7a3c2d27d268)

<mark>**Cleaned Patients Dataset**</mark>

In [18]:
from pyspark.sql.functions import (
    trim,
    when,
    upper,
    col,
    coalesce,
    lit,
    to_date,
    year,
    current_date
)

df_patients_clean = df_patients \
    .withColumn(
        "patient_name",
        trim(col("patient_name"))
    ) \
    .withColumn(
        "gender",
        when(
            upper(trim(col("gender"))) == "M",
            "Male"
        )
        .when(
            upper(trim(col("gender"))) == "F",
            "Female"
        )
        .otherwise(trim(col("gender")))
    ) \
    .withColumn(
        "date_of_birth",
        to_date(col("date_of_birth"), "M/d/yyyy")
    ) \
    .withColumn(
        "registration_date",
        to_date(col("registration_date"), "M/d/yyyy")
    ) \
    .withColumn(
        "city",
        trim(col("city"))
    ) \
    .withColumn(
        "state",
        trim(col("state"))
    ) \
    .withColumn(
        "blood_group",
        coalesce(trim(col("blood_group")), lit("Unknown"))
    ) \
    .withColumn(
        "email",
        coalesce(trim(col("email")), lit("Unknown"))
    ) \
    .withColumn(
        "age",
        year(current_date()) - year(col("date_of_birth"))
    ) \
    .withColumn(
        "age_group",
        when(col("age") < 18, "Child")
        .when(col("age").between(18, 35), "Young Adult")
        .when(col("age").between(36, 60), "Adult")
        .when(col("age").between(61, 75), "Senior Citizen")
        .when(col("age").between(76, 100), "Super Senior Citizen")
        .otherwise("Centenarian")
    ) \
    .dropDuplicates(["patient_id"])


df_patients_clean = df_patients_clean.select(
    "patient_id",
    "patient_name",
    "gender",
    "date_of_birth",
    "city",
    "state",
    "blood_group",
    "registration_date",
    "phone_number",
    "email",
    "age",
    "age_group"
)

display(df_patients_clean)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ddba0cdd-94fa-4973-aa9c-6e89a144f405)

<mark>**appointments.csv Read**</mark>

In [19]:
df_appointments = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://Healthcare_Analytics_Project@onelake.dfs.fabric.microsoft.com/lh_healthcare_bronze.Lakehouse/Files/appointments.csv")

display(df_appointments)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e4a04d9b-ac84-4417-9394-b03ef8cce6e3)

In [20]:
df_appointments.printSchema()

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 22, Finished, Available, Finished, False)

root
 |-- appointment_id: integer (nullable = true)
 |-- patient_id: integer (nullable = true)
 |-- doctor_id: integer (nullable = true)
 |-- hospital_id: integer (nullable = true)
 |-- scheduled_date: string (nullable = true)
 |-- appointment_date: string (nullable = true)
 |-- appointment_hour: integer (nullable = true)
 |-- appointment_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- waiting_time_minutes: integer (nullable = true)
 |-- consultation_duration_minutes: integer (nullable = true)
 |-- sms_reminder_sent: string (nullable = true)
 |-- date_diff: integer (nullable = true)



<mark>**Cleaned appointments dataset**</mark>

In [21]:
from pyspark.sql.functions import (
    trim,
    when,
    upper,
    col,
    coalesce,
    lit,
    to_date,
    datediff
)

df_appointments_clean = df_appointments \
    .withColumn(
        "scheduled_date",
        to_date(col("scheduled_date"), "M/d/yyyy")
    ) \
    .withColumn(
        "appointment_date",
        to_date(col("appointment_date"), "M/d/yyyy")
    ) \
    .withColumn(
        "appointment_type",
        trim(col("appointment_type"))
    ) \
    .withColumn(
        "status",
        when(
            upper(trim(col("status"))) == "COMPLETED",
            "Completed"
        )
        .when(
            upper(trim(col("status"))) == "CANCELLED",
            "Cancelled"
        )
        .when(
            upper(trim(col("status"))) == "NO SHOW",
            "No Show"
        )
        .otherwise(trim(col("status")))
    ) \
    .withColumn(
        "waiting_time_minutes",
        when(col("waiting_time_minutes") < 0, 0)
        .otherwise(col("waiting_time_minutes"))
    ) \
    .withColumn(
        "consultation_duration_minutes",
        when(col("consultation_duration_minutes") < 0, 0)
        .otherwise(col("consultation_duration_minutes"))
    ) \
    .withColumn(
        "sms_reminder_sent",
        when(
            upper(trim(col("sms_reminder_sent"))).isin("TRUE", "YES"),
            "Yes"
        ).otherwise("No")
    ) \
    .withColumn(
        "date_diff",
        datediff(
            col("appointment_date"),
            col("scheduled_date")
        )
    ) \
    .withColumn(
        "no_show_flag",
        when(col("status") == "No Show", 1)
        .otherwise(0)
    ) \
    .withColumn(
        "cancelled_flag",
        when(col("status") == "Cancelled", 1)
        .otherwise(0)
    )

# Maintain original CSV column order + derived columns

df_appointments_clean = df_appointments_clean.select(
    "appointment_id",
    "patient_id",
    "doctor_id",
    "hospital_id",
    "scheduled_date",
    "appointment_date",
    "appointment_hour",
    "appointment_type",
    "status",
    "waiting_time_minutes",
    "consultation_duration_minutes",
    "sms_reminder_sent",
    "date_diff",
    "no_show_flag",
    "cancelled_flag"
)

display(df_appointments_clean)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1340c1d7-e1a3-43fb-9cf3-27c6f8c79d4d)

<mark>**billing.csv Read**</mark>

In [22]:
df_billing = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://Healthcare_Analytics_Project@onelake.dfs.fabric.microsoft.com/lh_healthcare_bronze.Lakehouse/Files/billing.csv")

display(df_billing)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c1a689b4-5db9-4e1f-a340-ed0f0d9bf8c4)

In [23]:
df_billing.printSchema()

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 25, Finished, Available, Finished, False)

root
 |-- bill_id: integer (nullable = true)
 |-- appointment_id: integer (nullable = true)
 |-- patient_id: integer (nullable = true)
 |-- hospital_id: integer (nullable = true)
 |-- billing_date: string (nullable = true)
 |-- consultation_fee: integer (nullable = true)
 |-- medicine_cost: integer (nullable = true)
 |-- lab_test_cost: integer (nullable = true)
 |-- room_charge: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- total_amount: integer (nullable = true)



<mark>**Cleaned billing dataset**</mark>

In [24]:
from pyspark.sql.functions import (
    trim,
    when,
    upper,
    col,
    coalesce,
    lit,
    to_date,
    abs
)

df_billing_clean = df_billing \
    .withColumn(
        "billing_date",
        to_date(col("billing_date"), "M/d/yyyy")
    ) \
    .withColumn(
        "medicine_cost",
        coalesce(col("medicine_cost"), lit(0))
    ) \
    .withColumn(
        "lab_test_cost",
        coalesce(col("lab_test_cost"), lit(0))
    ) \
    .withColumn(
        "room_charge",
        coalesce(col("room_charge"), lit(0))
    ) \
    .withColumn(
        "payment_method",
        coalesce(trim(col("payment_method")), lit("Unknown"))
    ) \
    .withColumn(
        "payment_status",
        coalesce(trim(col("payment_status")), lit("Unknown"))
    ) \
    .withColumn(
        "consultation_fee",
        abs(col("consultation_fee"))
    ) \
    .withColumn(
        "medicine_cost",
        abs(col("medicine_cost"))
    ) \
    .withColumn(
        "lab_test_cost",
        abs(col("lab_test_cost"))
    ) \
    .withColumn(
        "room_charge",
        abs(col("room_charge"))
    ) \
    .withColumn(
        "total_amount",
        col("consultation_fee")
        + col("medicine_cost")
        + col("lab_test_cost")
        + col("room_charge")
    )

# Maintain original CSV column order

df_billing_clean = df_billing_clean.select(
    "bill_id",
    "appointment_id",
    "patient_id",
    "hospital_id",
    "billing_date",
    "consultation_fee",
    "medicine_cost",
    "lab_test_cost",
    "room_charge",
    "payment_method",
    "payment_status",
    "total_amount"
)

display(df_billing_clean)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ed66698d-d34f-4568-9e17-18bf822059bc)

<mark>**insurance_claims.csv Read**</mark>

In [25]:
df_insurance_claims = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://Healthcare_Analytics_Project@onelake.dfs.fabric.microsoft.com/lh_healthcare_bronze.Lakehouse/Files/insurance_claims.csv")

display(df_insurance_claims)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7c5abf68-8931-4520-a4a7-2367ee274c3d)

In [26]:
df_insurance_claims.printSchema()

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 28, Finished, Available, Finished, False)

root
 |-- claim_id: integer (nullable = true)
 |-- bill_id: integer (nullable = true)
 |-- patient_id: integer (nullable = true)
 |-- insurance_provider: string (nullable = true)
 |-- policy_type: string (nullable = true)
 |-- claim_amount: integer (nullable = true)
 |-- approved_amount: integer (nullable = true)
 |-- claim_status: string (nullable = true)
 |-- claim_date: string (nullable = true)
 |-- settlement_days: integer (nullable = true)



<mark>**Cleaned Insurance Claims Dataset**</mark>

In [27]:
from pyspark.sql.functions import (
    trim,
    when,
    upper,
    col,
    coalesce,
    lit,
    to_date,
    round
)

df_insurance_claims_clean = df_insurance_claims \
    .withColumn(
        "insurance_provider",
        coalesce(trim(col("insurance_provider")), lit("Unknown"))
    ) \
    .withColumn(
        "policy_type",
        trim(col("policy_type"))
    ) \
    .withColumn(
        "claim_status",
        trim(col("claim_status"))
    ) \
    .withColumn(
        "claim_date",
        to_date(col("claim_date"), "M/d/yyyy")
    ) \
    .withColumn(
        "approved_amount",
        coalesce(col("approved_amount"), lit(0))
    ) \
    .withColumn(
        "approved_amount",
        when(
            col("approved_amount") > col("claim_amount"),
            col("claim_amount")
        ).otherwise(col("approved_amount"))
    ) \
    .withColumn(
        "settlement_days",
        when(col("settlement_days") < 0, 0)
        .otherwise(col("settlement_days"))
    ) \
    .withColumn(
        "approval_rate",
        round(
            (col("approved_amount") / col("claim_amount")) * 100,
            2
        )
    ) \
    .withColumn(
        "claim_delay_bucket",
        when(col("settlement_days") <= 7, "Fast")
        .when(col("settlement_days").between(8, 15), "Moderate")
        .otherwise("Delayed")
    )

# Maintain original CSV column order + derived columns

df_insurance_claims_clean = df_insurance_claims_clean.select(
    "claim_id",
    "bill_id",
    "patient_id",
    "insurance_provider",
    "policy_type",
    "claim_amount",
    "approved_amount",
    "claim_status",
    "claim_date",
    "settlement_days",
    "approval_rate",
    "claim_delay_bucket"
)

display(df_insurance_claims_clean)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 44dd37ea-7993-4616-b54e-12627c1b4beb)

<mark>**lab_tests.csv Read**</mark>

In [28]:
df_lab_tests = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://Healthcare_Analytics_Project@onelake.dfs.fabric.microsoft.com/lh_healthcare_bronze.Lakehouse/Files/lab_tests.csv")

display(df_lab_tests)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7dd9ade2-7230-44dd-8584-0f3ce9d95373)

In [29]:
df_lab_tests.printSchema()

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 31, Finished, Available, Finished, False)

root
 |-- lab_test_id: integer (nullable = true)
 |-- appointment_id: integer (nullable = true)
 |-- patient_id: integer (nullable = true)
 |-- doctor_id: integer (nullable = true)
 |-- test_name: string (nullable = true)
 |-- test_date: string (nullable = true)
 |-- test_result: string (nullable = true)
 |-- test_cost: integer (nullable = true)
 |-- report_delivery_days: integer (nullable = true)



<mark>**Cleaned Lab Tests Dataset**</mark>

In [30]:
from pyspark.sql.functions import (
    trim,
    when,
    upper,
    col,
    coalesce,
    lit,
    to_date,
    abs
)

df_lab_tests_clean = df_lab_tests \
    .withColumn(
        "test_name",
        trim(col("test_name"))
    ) \
    .withColumn(
        "test_date",
        to_date(col("test_date"), "M/d/yyyy")
    ) \
    .withColumn(
        "test_result",
        coalesce(trim(col("test_result")), lit("Unknown"))
    ) \
    .withColumn(
        "test_cost",
        coalesce(col("test_cost"), lit(0))
    ) \
    .withColumn(
        "test_cost",
        abs(col("test_cost"))
    ) \
    .withColumn(
        "report_delivery_days",
        when(col("report_delivery_days") < 0, 0)
        .otherwise(col("report_delivery_days"))
    ) \
    .withColumn(
        "critical_flag",
        when(
            upper(trim(col("test_result"))).isin(
                "CRITICAL",
                "POSITIVE",
                "ABNORMAL"
            ),
            1
        ).otherwise(0)
    )

# Maintain original CSV column order + derived column

df_lab_tests_clean = df_lab_tests_clean.select(
    "lab_test_id",
    "appointment_id",
    "patient_id",
    "doctor_id",
    "test_name",
    "test_date",
    "test_result",
    "test_cost",
    "report_delivery_days",
    "critical_flag"
)

display(df_lab_tests_clean)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 85209317-c232-42e0-be37-08dfb9668a2b)

<mark>**patient_feedback.csv Read**</mark>

In [31]:
df_patient_feedback = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://Healthcare_Analytics_Project@onelake.dfs.fabric.microsoft.com/lh_healthcare_bronze.Lakehouse/Files/patient_feedback.csv")

display(df_patient_feedback)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 33, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2f633d89-9575-4624-9307-7b553f6f0239)

In [32]:
df_patient_feedback.printSchema()

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 34, Finished, Available, Finished, False)

root
 |-- feedback_id: integer (nullable = true)
 |-- appointment_id: integer (nullable = true)
 |-- patient_id: integer (nullable = true)
 |-- hospital_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- feedback_comment: string (nullable = true)
 |-- feedback_date: string (nullable = true)



<mark>**Cleaned Patient Feedback Dataset**</mark>

In [33]:
from pyspark.sql.functions import (
    trim,
    when,
    col,
    coalesce,
    lit,
    to_date
)

df_patient_feedback_clean = df_patient_feedback \
    .withColumn(
        "feedback_date",
        to_date(col("feedback_date"), "M/d/yyyy")
    ) \
    .withColumn(
        "rating",
        coalesce(col("rating"), lit(0))
    ) \
    .withColumn(
        "feedback_comment",
        coalesce(
            trim(col("feedback_comment")),
            lit("No Comment")
        )
    ) \
    .withColumn(
        "satisfaction_group",
        when(col("rating") >= 4, "Satisfied")
        .when(col("rating") == 3, "Neutral")
        .otherwise("Unsatisfied")
    )

# Maintain original CSV column order + derived column

df_patient_feedback_clean = df_patient_feedback_clean.select(
    "feedback_id",
    "appointment_id",
    "patient_id",
    "hospital_id",
    "rating",
    "feedback_comment",
    "feedback_date",
    "satisfaction_group"
)

display(df_patient_feedback_clean)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c68c3efe-65bb-4613-b4c5-f2748fc893d9)

In [34]:
df_hospitals_clean.write.mode("overwrite").saveAsTable(
    "silver_hospitals"
)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 36, Finished, Available, Finished, False)

In [35]:
df_departments_clean.write.mode("overwrite").saveAsTable(
    "silver_departments"
)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 37, Finished, Available, Finished, False)

In [36]:
df_doctors_clean.write.mode("overwrite").saveAsTable(
    "silver_doctors"
)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 38, Finished, Available, Finished, False)

In [37]:
df_patients_clean.write.mode("overwrite").saveAsTable(
    "silver_patients"
)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 39, Finished, Available, Finished, False)

In [38]:
df_appointments_clean.write.mode("overwrite").saveAsTable(
    "silver_appointments"
)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 40, Finished, Available, Finished, False)

In [39]:
df_billing_clean.write.mode("overwrite").saveAsTable(
    "silver_billing"
)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 41, Finished, Available, Finished, False)

In [40]:
df_insurance_claims_clean.write.mode("overwrite").saveAsTable(
    "silver_insurance_claims"
)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 42, Finished, Available, Finished, False)

In [41]:
df_lab_tests_clean.write.mode("overwrite").saveAsTable(
    "silver_lab_tests"
)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 43, Finished, Available, Finished, False)

In [42]:
df_patient_feedback_clean.write.mode("overwrite").saveAsTable(
    "silver_patient_feedback"
)

StatementMeta(, aecb6ee2-b288-40eb-a518-104bc046f0a9, 44, Finished, Available, Finished, False)